In [1]:
import polars as pl
import duckdb
from pathlib import Path

In [2]:
DATA_PATH = "../data/raw/indonesia_palm_oil_v1_2_5.csv"

print("Dataset:", DATA_PATH)
print("File tersedia:", Path(DATA_PATH).exists())

Dataset: ../data/raw/indonesia_palm_oil_v1_2_5.csv
File tersedia: True


In [3]:
df = pl.scan_csv(DATA_PATH)

print("Dataset berhasil dibaca dengan Polars Lazy API.")

Dataset berhasil dibaca dengan Polars Lazy API.


In [4]:
initial_rows = (
    df
    .select(pl.len().alias("jumlah_baris"))
    .collect()
    .item()
)

print("Jumlah baris sebelum cleaning:", initial_rows)

Jumlah baris sebelum cleaning: 1694907


In [5]:
summary = duckdb.sql(f"""
    SUMMARIZE SELECT *
    FROM read_csv_auto('{DATA_PATH}')
""")

summary

┌───────────────────────────────────────────────┬─────────────┬────────────────────────────────┬───────────────────────────┬───────────────┬─────────────────────┬────────────────────┬─────────────────────┬───────────────────────┬────────────────────────┬─────────┬─────────────────┐
│                  column_name                  │ column_type │              min               │            max            │ approx_unique │         avg         │        std         │         q25         │          q50          │          q75           │  count  │ null_percentage │
│                    varchar                    │   varchar   │            varchar             │          varchar          │     int64     │       varchar       │      varchar       │       varchar       │        varchar        │        varchar         │  int64  │  decimal(9,2)   │
├───────────────────────────────────────────────┼─────────────┼────────────────────────────────┼───────────────────────────┼───────────────┼───────────

In [6]:
type_check = df.select([
    pl.col("year"),
    pl.col("volume"),
    pl.col("oil_palm_ha"),
    pl.col("fob")
]).head(10).collect()

type_check

year,volume,oil_palm_ha,fob
i64,f64,str,str
2013,22.8,null,null
2013,3000.0,null,null
2013,1000.0,null,null
2013,1000.0,null,null
2013,500.0,null,null
2013,2500.0,null,null
2013,500.0,null,null
2013,1000.0,null,null
2013,500.0,null,null


In [7]:
for col in ["oil_palm_ha", "fob"]:
    print(f"\n===== {col} =====")
    
    result = (
        df
        .select(pl.col(col))
        .drop_nulls()
        .head(10)
        .collect()
    )
    
    print(result)


===== oil_palm_ha =====
shape: (10, 1)
┌────────────────────┐
│ oil_palm_ha        │
│ ---                │
│ str                │
╞════════════════════╡
│ 187.57247469148024 │
│ 31.28815257572648  │
│ 25.03052206058118  │
│ 3.1536589396105703 │
│ 242.3898299281281  │
│ 40.431998319954644 │
│ 32.34559865596371  │
│ 4.0577430208409995 │
│ 55.15159285937578  │
│ 9.19959847529204   │
└────────────────────┘

===== fob =====
shape: (10, 1)
┌────────────────────┐
│ fob                │
│ ---                │
│ str                │
╞════════════════════╡
│ 418247.53909158794 │
│ 55909.44031368691  │
│ 50761.623380925856 │
│ 5362.123618703253  │
│ 508539.28103506553 │
│ 67979.23220767397  │
│ 61720.09885431983  │
│ 6491.610619615029  │
│ 109533.42826732733 │
│ 14641.933538608626 │
└────────────────────┘


In [8]:
numeric_string_columns = [
    "oil_palm_ha",
    "fob",
    "total_emission_exposure",
    "net_emission_luc_exposure",
    "emission_subsidence_exposure",
    "gross_emission_luc_exposure",
    "emissions_fire_on_peat_exposure"
]

for col in numeric_string_columns:
    invalid = (
        df
        .filter(
            pl.col(col).is_not_null() &
            pl.col(col).cast(pl.Float64, strict=False).is_null()
        )
        .select(pl.col(col))
        .unique()
        .head(20)
        .collect()
    )

    print(f"\n===== {col} =====")
    print(invalid)


===== oil_palm_ha =====
shape: (0, 1)
┌─────────────┐
│ oil_palm_ha │
│ ---         │
│ str         │
╞═════════════╡
└─────────────┘

===== fob =====
shape: (0, 1)
┌─────┐
│ fob │
│ --- │
│ str │
╞═════╡
└─────┘

===== total_emission_exposure =====
shape: (0, 1)
┌─────────────────────────┐
│ total_emission_exposure │
│ ---                     │
│ str                     │
╞═════════════════════════╡
└─────────────────────────┘

===== net_emission_luc_exposure =====
shape: (0, 1)
┌───────────────────────────┐
│ net_emission_luc_exposure │
│ ---                       │
│ str                       │
╞═══════════════════════════╡
└───────────────────────────┘

===== emission_subsidence_exposure =====
shape: (0, 1)
┌──────────────────────────────┐
│ emission_subsidence_exposure │
│ ---                          │
│ str                          │
╞══════════════════════════════╡
└──────────────────────────────┘

===== gross_emission_luc_exposure =====
shape: (0, 1)
┌────────────────────────

In [9]:
year_values = (
    df
    .select("year")
    .unique()
    .sort("year")
    .collect()
)

year_values

year
i64
2013
2014
2015
2016
2017
2018
2019
2020
2021


In [13]:
duplicate_check = duckdb.sql(f"""
    SELECT
        COUNT(*) AS total_baris,
        COUNT(*) AS baris_unik
    FROM read_csv_auto('{DATA_PATH}')
""")

duplicate_check

┌─────────────┬────────────┐
│ total_baris │ baris_unik │
│    int64    │   int64    │
├─────────────┼────────────┤
│     1694907 │    1694907 │
└─────────────┴────────────┘

In [14]:
missing_check = (
    df
    .select([
        pl.col(c).null_count().alias(c)
        for c in df.collect_schema().names()
    ])
    .collect()
    .transpose(
        include_header=True,
        header_name="column",
        column_names=["missing_count"]
    )
    .with_columns(
        (pl.col("missing_count") / initial_rows * 100)
        .round(2)
        .alias("missing_percentage")
    )
    .sort("missing_count", descending=True)
)

missing_check

column,missing_count,missing_percentage
str,u32,f64
"""total_emission_exposure""",1025290,60.49
"""net_emission_luc_exposure""",1025290,60.49
"""emission_subsidence_exposure""",1025290,60.49
"""gross_emission_luc_exposure""",1025290,60.49
"""emissions_fire_on_peat_exposur…",1025290,60.49
…,…,…
"""province_of_production_trase_i…",0,0.0
"""kabupaten_of_production_trase_…",0,0.0
"""country_of_production_trase_id""",0,0.0


In [15]:
emission_missing_by_product = (
    df
    .group_by("product_type")
    .agg([
        pl.len().alias("total_baris"),
        pl.col("total_emission_exposure").null_count().alias("missing_total_emission"),
        pl.col("net_emission_luc_exposure").null_count().alias("missing_net_emission"),
        pl.col("emission_subsidence_exposure").null_count().alias("missing_subsidence"),
        pl.col("gross_emission_luc_exposure").null_count().alias("missing_gross_emission"),
        pl.col("emissions_fire_on_peat_exposure").null_count().alias("missing_fire_peat")
    ])
    .collect()
)

emission_missing_by_product

product_type,total_baris,missing_total_emission,missing_net_emission,missing_subsidence,missing_gross_emission,missing_fire_peat
str,u32,u32,u32,u32,u32,u32
"""PALM OIL""",168053,129915,129915,129915,129915,129915
"""REFINED PALM OIL""",1526854,895375,895375,895375,895375,895375


In [16]:
emission_missing_by_year = (
    df
    .group_by("year")
    .agg([
        pl.len().alias("total_baris"),
        pl.col("total_emission_exposure").null_count().alias("missing_emission")
    ])
    .with_columns(
        (
            pl.col("missing_emission") /
            pl.col("total_baris") * 100
        )
        .round(2)
        .alias("missing_percentage")
    )
    .sort("year")
    .collect()
)

emission_missing_by_year

year,total_baris,missing_emission,missing_percentage
i64,u32,u32,f64
2013,2095,2095,100.0
2014,2056,2056,100.0
2015,2038,2038,100.0
2016,2213,2213,100.0
2017,1831,1831,100.0
2018,247986,247986,100.0
2019,318343,318343,100.0
2020,448728,448728,100.0
2021,385389,0,0.0


In [17]:
small_missing = (
    missing_check
    .filter(
        (pl.col("missing_count") > 0) &
        (pl.col("missing_percentage") < 10)
    )
)

small_missing

column,missing_count,missing_percentage
str,u32,f64
"""fob""",10957,0.65
"""palm_oil_deforestation_10_year…",10233,0.6
"""palm_oil_deforestation_annual_…",10233,0.6
"""oil_palm_ha""",10233,0.6


In [18]:
small_missing_pattern = (
    df
    .select([
        pl.col("fob").is_null().alias("fob_missing"),
        pl.col("palm_oil_deforestation_10_year_total_exposure")
        .is_null()
        .alias("deforestation_10y_missing"),
        pl.col("palm_oil_deforestation_annual_exposure")
        .is_null()
        .alias("deforestation_annual_missing"),
        pl.col("oil_palm_ha")
        .is_null()
        .alias("oil_palm_ha_missing")
    ])
    .group_by([
        "fob_missing",
        "deforestation_10y_missing",
        "deforestation_annual_missing",
        "oil_palm_ha_missing"
    ])
    .agg(
        pl.len().alias("jumlah_baris")
    )
    .sort("jumlah_baris", descending=True)
    .collect()
)

small_missing_pattern

fob_missing,deforestation_10y_missing,deforestation_annual_missing,oil_palm_ha_missing,jumlah_baris
bool,bool,bool,bool,u32
false,false,false,false,1683950
true,true,true,true,10233
true,false,false,false,724


In [19]:
numeric_columns = [
    "oil_palm_ha",
    "fob",
    "total_emission_exposure",
    "net_emission_luc_exposure",
    "emission_subsidence_exposure",
    "gross_emission_luc_exposure",
    "emissions_fire_on_peat_exposure"
]

df_clean = (
    df
    .with_columns([
        pl.col(col)
        .cast(pl.Float64, strict=False)
        .alias(col)
        for col in numeric_columns
    ])
)

print("Konversi tipe data selesai.")

Konversi tipe data selesai.


In [20]:
type_check_clean = (
    df_clean
    .select([
        pl.col(col).alias(col)
        for col in numeric_columns
    ])
    .collect_schema()
)

type_check_clean

Schema([('oil_palm_ha', Float64),
        ('fob', Float64),
        ('total_emission_exposure', Float64),
        ('net_emission_luc_exposure', Float64),
        ('emission_subsidence_exposure', Float64),
        ('gross_emission_luc_exposure', Float64),
        ('emissions_fire_on_peat_exposure', Float64)])

In [21]:
conversion_check = (
    df_clean
    .select([
        pl.col(col).null_count().alias(f"{col}_null")
        for col in numeric_columns
    ])
    .collect()
)

conversion_check

oil_palm_ha_null,fob_null,total_emission_exposure_null,net_emission_luc_exposure_null,emission_subsidence_exposure_null,gross_emission_luc_exposure_null,emissions_fire_on_peat_exposure_null
u32,u32,u32,u32,u32,u32,u32
10233,10957,1025290,1025290,1025290,1025290,1025290


In [22]:
negative_check = (
    df_clean
    .select([
        (pl.col(col) < 0).sum().alias(col)
        for col in numeric_columns
    ])
    .collect()
)

negative_check

oil_palm_ha,fob,total_emission_exposure,net_emission_luc_exposure,emission_subsidence_exposure,gross_emission_luc_exposure,emissions_fire_on_peat_exposure
u32,u32,u32,u32,u32,u32,u32
0,0,16456,34557,0,0,0


In [23]:
negative_values = (
    df_clean
    .select([
        "year",
        "product_type",
        "province_of_production",
        "total_emission_exposure",
        "net_emission_luc_exposure"
    ])
    .filter(
        (pl.col("total_emission_exposure") < 0) |
        (pl.col("net_emission_luc_exposure") < 0)
    )
    .head(20)
    .collect()
)

negative_values

year,product_type,province_of_production,total_emission_exposure,net_emission_luc_exposure
i64,str,str,f64,f64
2021,"""PALM OIL""","""KALIMANTAN BARAT""",-0.019469,-0.019469
2021,"""PALM OIL""","""KALIMANTAN TENGAH""",-86.264885,-86.264885
2021,"""PALM OIL""","""KALIMANTAN TIMUR""",-21.923635,-21.923635
2021,"""PALM OIL""","""KALIMANTAN TIMUR""",-34.656706,-34.656706
2021,"""PALM OIL""","""ACEH""",8.669942,-0.031472
…,…,…,…,…
2021,"""PALM OIL""","""LAMPUNG""",10.167042,-0.134383
2021,"""PALM OIL""","""PAPUA TENGAH""",0.397837,-0.006225
2021,"""PALM OIL""","""SUMATERA BARAT""",0.265322,-0.000401


In [24]:
negative_by_year = (
    df_clean
    .filter(
        (pl.col("total_emission_exposure") < 0) |
        (pl.col("net_emission_luc_exposure") < 0)
    )
    .group_by("year")
    .agg([
        (pl.col("total_emission_exposure") < 0)
        .sum()
        .alias("negative_total_emission"),
        
        (pl.col("net_emission_luc_exposure") < 0)
        .sum()
        .alias("negative_net_emission")
    ])
    .sort("year")
    .collect()
)

negative_by_year

year,negative_total_emission,negative_net_emission
i64,u32,u32
2021,12452,27367
2022,4004,7190


In [26]:
df_clean = (
    df_clean
    .with_columns([
        pl.col("palm_oil_deforestation_10_year_total_exposure")
        .cast(pl.Float64, strict=False),

        pl.col("palm_oil_deforestation_annual_exposure")
        .cast(pl.Float64, strict=False)
    ])
)

print("Kolom deforestation berhasil dikonversi.")

Kolom deforestation berhasil dikonversi.


In [27]:
zero_check = (
    df_clean
    .select([
        (pl.col(col) == 0).sum().alias(col)
        for col in [
            "volume",
            "oil_palm_ha",
            "fob",
            "palm_oil_deforestation_10_year_total_exposure",
            "palm_oil_deforestation_annual_exposure"
        ]
    ])
    .collect()
)

zero_check

volume,oil_palm_ha,fob,palm_oil_deforestation_10_year_total_exposure,palm_oil_deforestation_annual_exposure
u32,u32,u32,u32,u32
0,6061,37292,895465,1324604


In [29]:
clean_numeric_summary = (
    df_clean
    .select([
        "year",
        "volume",
        "oil_palm_ha",
        "fob",
        "palm_oil_deforestation_10_year_total_exposure",
        "palm_oil_deforestation_annual_exposure",
        "total_emission_exposure",
        "net_emission_luc_exposure",
        "emission_subsidence_exposure",
        "gross_emission_luc_exposure",
        "emissions_fire_on_peat_exposure"
    ])
    .describe()
)

clean_numeric_summary

statistic,year,volume,oil_palm_ha,fob,palm_oil_deforestation_10_year_total_exposure,palm_oil_deforestation_annual_exposure,total_emission_exposure,net_emission_luc_exposure,emission_subsidence_exposure,gross_emission_luc_exposure,emissions_fire_on_peat_exposure
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",1.694907e6,1.694907e6,1.684674e6,1.68395e6,1.684674e6,1.684674e6,669617.0,669617.0,669617.0,669617.0,669617.0
"""null_count""",0.0,0.0,10233.0,10957.0,10233.0,10233.0,1.02529e6,1.02529e6,1.02529e6,1.02529e6,1.02529e6
"""mean""",2020.051917,226.346597,47.36969,70013.442567,4.210289,0.101668,482.627556,26.855007,439.224326,69.791003,16.548223
"""std""",1.35405,15280.810851,1357.998033,852230.163937,223.557235,9.172228,24424.125906,2159.253975,22901.583184,4259.561017,2291.923455
"""min""",2013.0,5.0414e-11,0.0,0.0,0.0,0.0,-31772.236169,-225993.363627,0.0,0.0,0.0
"""25%""",2019.0,0.152118,0.059135,90.04737,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""50%""",2020.0,2.12001,0.765239,1444.66696,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""75%""",2021.0,21.216855,7.340382,14547.571718,0.038487,0.0,4.430601,0.0,1.979799,0.0,0.0
"""max""",2022.0,1.1605e7,303954.785084,3.1002e8,62725.929197,3772.782719,6.2156e6,839686.605971,5.3127e6,1.2798e6,1.1258e6


In [30]:
outlier_columns = [
    "volume",
    "oil_palm_ha",
    "fob",
    "palm_oil_deforestation_10_year_total_exposure",
    "palm_oil_deforestation_annual_exposure"
]

outlier_summary = []

for col in outlier_columns:
    stats = (
        df_clean
        .select([
            pl.col(col).quantile(0.25).alias("Q1"),
            pl.col(col).quantile(0.75).alias("Q3")
        ])
        .collect()
    )

    q1 = stats["Q1"][0]
    q3 = stats["Q3"][0]
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    count_outlier = (
        df_clean
        .filter(
            (pl.col(col) < lower) |
            (pl.col(col) > upper)
        )
        .select(pl.len())
        .collect()
        .item()
    )

    outlier_summary.append({
        "column": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": count_outlier
    })

outlier_summary = pl.DataFrame(outlier_summary)

outlier_summary

column,Q1,Q3,IQR,lower_bound,upper_bound,outlier_count
str,f64,f64,f64,f64,f64,i64
"""volume""",0.152118,21.216855,21.064737,-31.444988,52.813961,281597
"""oil_palm_ha""",0.059135,7.340382,7.281247,-10.862735,18.262251,277386
"""fob""",90.04737,14547.571718,14457.524347,-21596.23915,36233.858238,279070
"""palm_oil_deforestation_10_year…",0.0,0.038487,0.038487,-0.057731,0.096218,338785
"""palm_oil_deforestation_annual_…",0.0,0.0,0.0,0.0,0.0,360070


In [31]:
extreme_values = (
    df_clean
    .select([
        "year",
        "product_type",
        "province_of_production",
        "kabupaten_of_production",
        "mill",
        "volume",
        "oil_palm_ha",
        "fob"
    ])
    .sort("volume", descending=True)
    .head(20)
    .collect()
)

extreme_values

year,product_type,province_of_production,kabupaten_of_production,mill,volume,oil_palm_ha,fob
i64,str,str,str,str,f64,f64,f64
2017,"""PALM OIL""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""",1.1605e7,null,null
2016,"""PALM OIL""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""",8.8145e6,null,null
2013,"""PALM OIL""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""",8.4254e6,null,null
2014,"""PALM OIL""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""",6.3537e6,null,null
2015,"""PALM OIL""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""",5.2897e6,null,null
…,…,…,…,…,…,…,…
2018,"""PALM OIL""","""KALIMANTAN TENGAH""","""KOTAWARINGIN TIMUR""","""UNKNOWN""",729545.536683,191018.849338,null
2015,"""PALM OIL""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""",700771.658,null,null
2022,"""PALM OIL""","""KALIMANTAN TIMUR""","""KUTAI TIMUR""","""UNKNOWN""",646974.992603,209364.371834,0.0


In [32]:
unknown_check = (
    df_clean
    .select([
        (pl.col("province_of_production") == "UNKNOWN")
        .sum()
        .alias("province_unknown"),

        (pl.col("kabupaten_of_production") == "UNKNOWN")
        .sum()
        .alias("kabupaten_unknown"),

        (pl.col("mill") == "UNKNOWN")
        .sum()
        .alias("mill_unknown"),

        (pl.col("refinery") == "UNKNOWN")
        .sum()
        .alias("refinery_unknown"),

        (pl.col("exporter") == "UNKNOWN")
        .sum()
        .alias("exporter_unknown"),

        (pl.col("importer") == "UNKNOWN")
        .sum()
        .alias("importer_unknown")
    ])
    .collect()
)

unknown_check

province_unknown,kabupaten_unknown,mill_unknown,refinery_unknown,exporter_unknown,importer_unknown
u32,u32,u32,u32,u32,u32
10233,10233,814388,787967,1691,245028


In [33]:
unknown_missing_check = (
    df_clean
    .select([
        (pl.col("province_of_production") == "UNKNOWN")
        .alias("province_unknown"),

        pl.col("oil_palm_ha").is_null()
        .alias("oil_palm_ha_missing"),

        pl.col("fob").is_null()
        .alias("fob_missing")
    ])
    .group_by([
        "province_unknown",
        "oil_palm_ha_missing",
        "fob_missing"
    ])
    .agg(
        pl.len().alias("jumlah_baris")
    )
    .sort("jumlah_baris", descending=True)
    .collect()
)

unknown_missing_check

province_unknown,oil_palm_ha_missing,fob_missing,jumlah_baris
bool,bool,bool,u32
false,false,false,1683950
true,true,true,10233
false,false,true,724


In [34]:
df_final = (
    df_clean
    .with_columns([
        pl.col("year").cast(pl.Int64),
        pl.col("volume").cast(pl.Float64),
        pl.col("oil_palm_ha").cast(pl.Float64),
        pl.col("fob").cast(pl.Float64),
        pl.col("palm_oil_deforestation_10_year_total_exposure").cast(pl.Float64),
        pl.col("palm_oil_deforestation_annual_exposure").cast(pl.Float64),
        pl.col("total_emission_exposure").cast(pl.Float64),
        pl.col("net_emission_luc_exposure").cast(pl.Float64),
        pl.col("emission_subsidence_exposure").cast(pl.Float64),
        pl.col("gross_emission_luc_exposure").cast(pl.Float64),
        pl.col("emissions_fire_on_peat_exposure").cast(pl.Float64)
    ])
)

print("Dataset cleaned final berhasil dibuat.")

Dataset cleaned final berhasil dibuat.


In [35]:
final_validation = {
    "baris_awal": initial_rows,
    "baris_akhir": df_final.select(pl.len()).collect().item(),
    "jumlah_kolom": len(df_final.collect_schema().names())
}

final_validation

{'baris_awal': 1694907, 'baris_akhir': 1694907, 'jumlah_kolom': 34}

In [36]:
final_schema = df_final.collect_schema()

final_schema

Schema([('year', Int64),
        ('country_of_production', String),
        ('product_type', String),
        ('forest_500_palm_oil', String),
        ('zero_deforestation_indonesia_palm_oil', String),
        ('province_of_production', String),
        ('kabupaten_of_production', String),
        ('mill', String),
        ('mill_trase_id', String),
        ('mill_group', String),
        ('refinery', String),
        ('refinery_trase_id', String),
        ('refinery_group', String),
        ('exporter', String),
        ('exporter_group', String),
        ('port_of_export', String),
        ('importer', String),
        ('importer_group', String),
        ('country_of_first_import', String),
        ('economic_bloc', String),
        ('province_of_production_trase_id', String),
        ('kabupaten_of_production_trase_id', String),
        ('country_of_production_trase_id', String),
        ('country_of_first_import_trase_id', String),
        ('palm_oil_deforestation_10_year_total_exp

In [37]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Folder processed:", PROCESSED_DIR)

Folder processed: ..\data\processed


In [38]:
OUTPUT_PATH = PROCESSED_DIR / "indonesia_palm_oil_cleaned.parquet"

df_final.sink_parquet(OUTPUT_PATH)

print("Dataset cleaned berhasil disimpan.")
print("Lokasi:", OUTPUT_PATH)

Dataset cleaned berhasil disimpan.
Lokasi: ..\data\processed\indonesia_palm_oil_cleaned.parquet


In [ ]:
output_file = Path(OUTPUT_PATH)

print("File tersedia:", output_file.exists())

if output_file.exists():
    print("Ukuran file:", round(output_file.stat().st_size / (1024**2), 2), "MB")

File tersedia: True
Ukuran file: 47.5 MB


In [40]:
df_parquet = pl.scan_parquet(OUTPUT_PATH)

parquet_validation = {
    "jumlah_baris": df_parquet.select(pl.len()).collect().item(),
    "jumlah_kolom": len(df_parquet.collect_schema().names())
}

print("Validasi dataset Parquet:")
print(parquet_validation)

Validasi dataset Parquet:
{'jumlah_baris': 1694907, 'jumlah_kolom': 34}
